
# 練習問題: べき乗法で「すごろく」の定常分布を求める

## 目標

`S` 個のマスが輪になった「すごろく」を, ずっと遊び続けると **どのマスに一番よく止まるか** を求める。これは Web ページの重要度を測る **PageRank の遊べる版**。手法はやはり **べき乗法 (power iteration)**: 遷移確率の行列を繰り返し掛けると **定常分布** に収束する。中身は **行列ベクトル積 (matvec)** と **総和 (reduction)** の繰り返しで, これまで学んだ並列化がそのまま使える。

## ルールとしくみ

- マス `0..S-1` が輪になっている。マス `s` でサイコロ (1..6) を振ると, それぞれ確率 1/6 で `(s+roll) mod S` へ進む。
- さらに数か所に **ワープ (はしご/すべり台)** がある: ある `from` マスに着くと即座に `to` マスへ飛ばされる。これがあると分布が **一様でなくなる** (人気のマスができる)。
- 1ターンの遷移確率を **行列 `M`** にまとめる: `M[t][s]` = マス `s` から `t` へ1ターンで移る確率。乱数は不要で, 各 `s` についてサイコロの目 1..6 を展開し, ワープで行き先を付け替えれば決定的に作れる (行優先なので `M[t*S + s]`)。
- **定常分布 pi**: `pi_new[t] = Σ_s M[t][s] pi[s]` を計算して合計が 1 になるよう正規化, を繰り返すと収束する。これがべき乗法 (この行列の最大固有値は 1)。

## 課題

`markov.cpp` (または `markov.f90`) は, 遷移行列 `M` を構築し, 一様分布 `pi=1/S` から始めて `pi_new[t]=Σ_s M[t][s]pi[s]` を反復する。

並列化すべき2つのカーネルが `TODO` になっている:

1. **matvec** (`pi_new[t] = Σ_s M[t][s] pi[s]`): 行 `t` の外側ループ。各 `t` は独立で内側の和はローカルなので, 競合もリダクションも不要。
   - C++: `#pragma omp parallel for`
   - Fortran: `!$omp parallel do private(s_, sm)` … `!$omp end parallel do`
2. **正規化の総和** (`total = Σ_t pi_new[t]`): リダクション。
   - C++: `#pragma omp parallel for reduction(+:total)`
   - Fortran: `!$omp parallel do reduction(+:total)` … `!$omp end parallel do`

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore markov.cpp -o markov.exe

# Fortran
nvfortran -fast -mp=multicore markov.f90 -o markov.exe
```

第1引数でマスの数 `S`, 第2引数で収束しきい値 `tol`, 第3引数で最大反復回数。

```
OMP_NUM_THREADS=4 ./markov.exe 1000 1e-10 100000
```

## 期待される結果

```
S=1000, iters=..., sum=1.0000000000
最も止まりやすいマス=1 (確率 0.002414), 一様なら 1/S=0.001000
上位3マス: 1(0.002414), 500(0.002353), 333(0.001752)
```

- `sum(pi)` が 1.0 になる (確率分布になっている)。
- ワープが無ければ分布は完全に一様 (どのマスも `1/S`) になるはず。**ワープがあるおかげで, ワープの行き先のマス (上の例では 1 や 500) の確率が明らかに高くなる** ── そこに「長居しやすい」。
- べき乗法の反復は数千回程度で収束する。`S` を大きくすると行列 `M` (S×S) の matvec が重くなり, 並列化の効果 (台数効果) が見えやすい。「性能を比べる」セルで測ってみよ。
- (発展) ワープの場所を変えて分布がどう変わるか試す, matvec を GPU にオフロードする, など。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ markov.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (未使用だが慣例として置いておく): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

/* ワープ (はしご/すべり台) の行き先を返す。from に該当しなければ d をそのまま返す。 */
static inline int warp(int d, int S) {
  if (d == 3)       return S / 2;
  if (d == S / 4)   return S - 2;
  if (d == S/2 + 5) return 1;
  if (d == S - 7)   return S / 3;
  return d;
}

int main(int argc, char ** argv) {
  int    S     = (argc > 1 ? atoi(argv[1]) : 1000);   /* マスの数 (0..S-1 の輪) */
  double tol   = (argc > 2 ? atof(argv[2]) : 1e-10);
  int    maxit = (argc > 3 ? atoi(argv[3]) : 100000);

  /* 遷移行列 M (密) を構築。M[t*S + s] = マス s から t へ1ターンで移る確率。
     各 s について サイコロ 1..6 を振り d=(s+roll)%S, ワープがあれば飛ばす。 */
  double * M = (double *)calloc((size_t)S * S, sizeof(double));
  for (int s = 0; s < S; s++)
    for (int roll = 1; roll <= 6; roll++) {
      int d = warp((s + roll) % S, S);
      M[(size_t)d * S + s] += 1.0 / 6.0;
    }

  double * pi  = (double *)malloc(sizeof(double) * S);
  double * pin = (double *)malloc(sizeof(double) * S);
  for (int s = 0; s < S; s++) pi[s] = 1.0 / S;        /* 一様分布から開始 */

  /* べき乗法: 遷移行列を繰り返し掛けると定常分布に収束する (最大固有値 = 1)。 */
  int it;
  double t0 = omp_get_wtime();
  for (it = 0; it < maxit; it++) {
    // TODO: 行 t ごとの行列ベクトル積を #pragma omp parallel for で並列化せよ (各 t は独立).
    for (int t = 0; t < S; t++) {
      double sum = 0.0;
      for (int s = 0; s < S; s++) sum += M[(size_t)t * S + s] * pi[s];
      pin[t] = sum;
    }
    double total = 0.0;
    // TODO: 総和を #pragma omp parallel for reduction(+:total) で並列化せよ.
    for (int t = 0; t < S; t++) total += pin[t];
    double diff = 0.0;
    for (int t = 0; t < S; t++) {
      pin[t] /= total;                                 /* 正規化 (sum=1) */
      double e = fabs(pin[t] - pi[t]); if (e > diff) diff = e;
      pi[t] = pin[t];
    }
    if (diff < tol) { it++; break; }
  }
  double elapsed = omp_get_wtime() - t0;

  /* 検算: sum(pi), 最も止まりやすいマスとその確率, 上位3マス */
  double sum = 0.0;
  for (int s = 0; s < S; s++) sum += pi[s];
  int best = 0;
  for (int s = 1; s < S; s++) if (pi[s] > pi[best]) best = s;

  /* 上位3マスを単純に探す */
  int top[3] = {-1, -1, -1};
  for (int r = 0; r < 3; r++) {
    int b = -1;
    for (int s = 0; s < S; s++) {
      bool used = false;
      for (int q = 0; q < r; q++) if (top[q] == s) used = true;
      if (used) continue;
      if (b < 0 || pi[s] > pi[b]) b = s;
    }
    top[r] = b;
  }

  printf("S=%d, iters=%d, sum=%.10f\n", S, it, sum);
  printf("最も止まりやすいマス=%d (確率 %.6f), 一様なら 1/S=%.6f\n",
         best, pi[best], 1.0 / S);
  printf("上位3マス: %d(%.6f), %d(%.6f), %d(%.6f)\n",
         top[0], pi[top[0]], top[1], pi[top[1]], top[2], pi[top[2]]);
  printf("elapsed = %.3f sec\n", elapsed);
  free(M); free(pi); free(pin);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore markov.cpp -o markov_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./markov_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ markov.f90
module markov_mod
contains
  ! 状態を持たない乱数 (未使用だが慣例として置いておく): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01

  ! ワープ (はしご/すべり台) の行き先。from に該当しなければ d をそのまま返す。
  function warp(d, S) result(r)
    integer, intent(in) :: d, S
    integer :: r
    if (d == 3) then
       r = S / 2
    else if (d == S / 4) then
       r = S - 2
    else if (d == S/2 + 5) then
       r = 1
    else if (d == S - 7) then
       r = S / 3
    else
       r = d
    end if
  end function warp
end module markov_mod

program markov
  use markov_mod
  use omp_lib
  character(len=32) :: arg
  integer :: S, maxit, it, s_, t, roll, d, best, r, q, b
  logical :: used
  real(8) :: tol, total, diff, e, sm, t0, elapsed
  integer :: top(0:2)
  real(8), allocatable :: M(:,:), pi(:), pin(:)
  S = 1000; tol = 1d-10; maxit = 100000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) S
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) tol
  end if
  if (command_argument_count() >= 3) then
     call get_command_argument(3, arg); read (arg, *) maxit
  end if

  ! 遷移行列 M (密) を構築。M(t,s) = マス s から t へ1ターンで移る確率。
  ! 各 s についてサイコロ 1..6 を振り d=mod(s+roll,S), ワープがあれば飛ばす。
  allocate(M(0:S-1, 0:S-1), pi(0:S-1), pin(0:S-1))
  M = 0.0d0
  do s_ = 0, S - 1
     do roll = 1, 6
        d = warp(modulo(s_ + roll, S), S)
        M(d, s_) = M(d, s_) + 1.0d0 / 6.0d0
     end do
  end do

  do s_ = 0, S - 1
     pi(s_) = 1.0d0 / S          ! 一様分布から開始
  end do

  ! べき乗法: 遷移行列を繰り返し掛けると定常分布に収束する (最大固有値 = 1)。
  t0 = omp_get_wtime()
  do it = 1, maxit
     ! TODO: 行 t ごとの行列ベクトル積を !$omp parallel do で並列化せよ (各 t は独立).
     do t = 0, S - 1
        sm = 0.0d0
        do s_ = 0, S - 1
           sm = sm + M(t, s_) * pi(s_)
        end do
        pin(t) = sm
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
     total = 0.0d0
     ! TODO: 総和を !$omp parallel do reduction(+:total) で並列化せよ.
     do t = 0, S - 1
        total = total + pin(t)
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
     diff = 0.0d0
     do t = 0, S - 1
        pin(t) = pin(t) / total              ! 正規化 (sum=1)
        e = abs(pin(t) - pi(t)); if (e > diff) diff = e
        pi(t) = pin(t)
     end do
     if (diff < tol) exit
  end do
  elapsed = omp_get_wtime() - t0

  ! 検算: sum(pi), 最も止まりやすいマスとその確率, 上位3マス
  sm = 0.0d0
  do s_ = 0, S - 1
     sm = sm + pi(s_)
  end do
  best = 0
  do s_ = 1, S - 1
     if (pi(s_) > pi(best)) best = s_
  end do

  top = -1
  do r = 0, 2
     b = -1
     do s_ = 0, S - 1
        used = .false.
        do q = 0, r - 1
           if (top(q) == s_) used = .true.
        end do
        if (used) cycle
        if (b < 0) then
           b = s_
        else if (pi(s_) > pi(b)) then
           b = s_
        end if
     end do
     top(r) = b
  end do

  print "(a,i0,a,i0,a,f0.10)", "S=", S, ", iters=", min(it, maxit), ", sum=", sm
  print "(a,i0,a,f0.6,a,f0.6)", &
       "最も止まりやすいマス=", best, " (確率 ", pi(best), "), 一様なら 1/S=", 1.0d0 / S
  print "(a,i0,a,f0.6,a,i0,a,f0.6,a,i0,a,f0.6,a)", &
       "上位3マス: ", top(0), "(", pi(top(0)), "), ", top(1), "(", pi(top(1)), &
       "), ", top(2), "(", pi(top(2)), ")"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program markov

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore markov.f90 -o markov_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./markov_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:markov.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:markov.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:markov.cpp}